# 04 Video Evaluation

Evaluate the trained CNN-LSTM model on the test set and analyze performance metrics.

### Metrics Reported:
- Accuracy
- Precision, Recall, F1 Score
- Confusion Matrix
- ROC Curve

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import yaml
import json
import os

# Load Config
with open("../../configs/video_config.yaml", "r") as f:
    config = yaml.safe_load(f)

IMG_SIZE = config["model"]["frame_size"]
FRAME_COUNT = config["model"]["frame_count"]
BATCH_SIZE = config["training"]["batch_size"]
MODEL_PATH = Path(config["data"]["model_save_path"])
DATA_PATH = config["data"]["raw_path"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Model & Dataset Definition

In [2]:
class DeepfakeVideoModel(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.3):
        super(DeepfakeVideoModel, self).__init__()
        backbone = models.efficientnet_b0(weights=None)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1)
        )
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        x = x.view(batch_size * time_steps, C, H, W)
        features = self.feature_extractor(x).view(batch_size * time_steps, -1).view(batch_size, time_steps, -1)
        _, (h_n, _) = self.lstm(features)
        return self.classifier(h_n[-1])

class VideoFrameDataset(Dataset):
    def __init__(self, video_list, num_frames=32, transform=None):
        self.video_list = video_list
        self.num_frames = num_frames
        self.transform = transform
    def __len__(self): return len(self.video_list)
    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: cap.release(); return None
        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                if self.transform: frame = self.transform(frame)
                frames.append(frame)
        cap.release()
        return torch.stack(frames)
    def __getitem__(self, index):
        path, label = self.video_list[index]
        frames = self._get_frames(path)
        if frames is None: frames = torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
        return frames, torch.tensor(label, dtype=torch.float32)

## 2. Load Model & Prepare Evaluation

In [3]:
model = DeepfakeVideoModel(hidden_dim=config["model"]["hidden_dim"], dropout=config["model"]["dropout"]).to(device)
if MODEL_PATH.exists():
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("Model weights loaded.")
else:
    print("Error: Model file not found.")

def discover_dataset_parts(base_path):
    base_path = Path(base_path)
    if not base_path.exists(): return []
    return sorted([base_path / d for d in os.listdir(base_path) if d.startswith("dfdc_train_part_")], key=lambda x: int(x.name.split('_')[-1]))

def get_video_list(parts_paths):
    video_list = []
    for part_path in parts_paths:
        metadata_path = part_path / "metadata.json"
        if not metadata_path.exists(): continue
        with open(metadata_path, "r") as f: metadata = json.load(f)
        for filename, info in metadata.items():
            if (part_path / filename).exists():
                video_list.append((str(part_path / filename), 1 if info["label"] == "FAKE" else 0))
    return video_list

parts = discover_dataset_parts(DATA_PATH)
all_vids = get_video_list(parts)
test_vids = all_vids[-500:] # Use last 500 for eval

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_loader = DataLoader(VideoFrameDataset(test_vids, num_frames=FRAME_COUNT, transform=transform), batch_size=BATCH_SIZE, shuffle=False)

## 3. Evaluation Metrics

In [4]:
model.eval()
preds, targets = [], []
with torch.no_grad():
    for f, l in test_loader:
        outputs = model(f.to(device))
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds.extend(probs)
        targets.extend(l.numpy())

preds_bin = np.array(preds) >= 0.5
print("Classification Report:")
print(classification_report(targets, preds_bin))

cm = confusion_matrix(targets, preds_bin)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["REAL", "FAKE"], yticklabels=["REAL", "FAKE"])
plt.title("Confusion Matrix")
plt.show()